# BrandSuit AI — category classification and advertiser suitability

This notebook is the analytical write-up for a **text-only** YouTube category model plus brand allow-lists.

It is not a brand-safety detector. YouTube Trending has no violence or hate labels. The model predicts a contextual category from title and description; each advertiser then applies an allow-list and a threshold.

**How to read this file.** Mapping, split, and model specs live in `src/config.py`. Metrics live in `reports/evaluation.json` from `py -m src.evaluate`. This notebook does not re-fit the four candidates. If that JSON is missing, run the evaluator first rather than typing numbers in by hand.

**Problem in one sentence.** Should this advertiser run next to this video, given only the title and description, knowing that the costly error for a kids brand is approving content whose *mapped* category is excluded?


## 1. Data and assumptions

- One row is one unique `video_id`. Repeated trending days are dropped first so the same video cannot sit in train and holdout.
- `X` is `title + description`. `category_id` and channel name are labels or grouping keys, never features.
- Official YouTube names are grouped into six advertiser-context classes. Pets & Animals stay in Lifestyle & Interests: a pet vlog is lifestyle inventory, not STEM. That choice is in `src/config.py`. It is not the mapping that maximised macro-F1 (an earlier export had moved Pets into Education).
- The 80/20 stratified split (`random_state=42`) is a **development holdout**. It is reused for comparison, error analysis, threshold talk, and calibration. It is not a sealed test set. A new shuffle of these videos would not be independent confirmation.
- Policy metrics treat mapped YouTube categories as *proxies* for contextual suitability, not as human-verified placement labels.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import CATEGORY_MAPPING, CATEGORY_MAPPING_RATIONALE, EVALUATION_PATH, TARGET_CLASSES
from src.dataset import make_primary_split, prepare_dataset

print("Pets & Animals ->", CATEGORY_MAPPING["Pets & Animals"])
print()
print(CATEGORY_MAPPING_RATIONALE[:400], "...")

In [ ]:
df = prepare_dataset()
train, holdout, split_info = make_primary_split(df)
overlap = set(train["video_id"]).intersection(set(holdout["video_id"]))

print("unique videos", len(df))
print("train", split_info["n_train"], "holdout", split_info["n_holdout"], "overlap", len(overlap))
print()
print(df["y"].value_counts().reindex(TARGET_CLASSES))
print()
print("YouTube name vs mapped label (Pets should sit only in Lifestyle):")
print(pd.crosstab(df["youtube_name"], df["y"]).loc[["Pets & Animals", "Education", "Science & Technology"]])

## 2. Experiments

Four specs, identical split membership:

1. Majority-class baseline
2. Unigram logistic regression, no class weights
3. Unigram logistic regression, balanced weights
4. Unigram-plus-bigram logistic regression, balanced weights

The business screen is Kids & Family at threshold 0.55: incorrect approvals from **every** excluded class (News, Gaming, Lifestyle, Education), then retention of videos whose mapped label is allowed. A 1-2 video gap on this holdout is treated as a tie. Macro-F1 is reported; it is not the selection rule. Bigrams are not kept because they sound more advanced.


In [ ]:
import json

payload = json.loads(EVALUATION_PATH.read_text(encoding="utf-8"))
assert payload["status"] == "completed", "Run py -m src.evaluate first"

print("selected:", payload["selected_model"]["label"])
print()
rows = []
for key, model in payload["models"].items():
    c = model["classification"]
    k = model["policies"]["Kids & Family Brand"]["at_stated_threshold"]
    rows.append({
        "model": model["label"],
        "accuracy": round(c["accuracy"], 3),
        "macro_f1": round(c["macro_f1"], 3),
        "news_recall": round(c["news_recall"], 3),
        "news_approved": f"{k['news_approvals']}/{k['news_in_split']}",
        "excluded_approved": f"{k['incorrect_approvals']}/{k['n_truly_excluded']}",
        "allowed_retained": f"{k['n_retained_allowed']}/{k['n_truly_allowed']}",
        "selected": key == payload["selected_model"]["key"],
    })
pd.DataFrame(rows)

In [ ]:
selected = payload["models"][payload["selected_model"]["key"]]
labels = selected["classification"]["confusion_matrix"]["labels"]
matrix = selected["classification"]["confusion_matrix"]["matrix"]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Mapped label")
ax.set_title("Selected model — development holdout")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

pd.DataFrame(selected["classification"]["per_class"]).T

## 3. Errors and label noise

The unweighted unigram model approved several real News videos for a kids allow-list. Balanced class weights pulled those `p_allow` scores below 0.55.

Remaining Kids & Family approvals from excluded mapped labels are mostly Lifestyle videos that *look* like sports or music (YouTube category noise) plus a few Gaming titles (true policy leaks). Evaluating against mapped labels over-states some errors and still misses that Gaming is a real kids-brand leak.


In [ ]:
print("News leaks: unweighted model vs selected (Kids & Family 0.55)")
for row in payload["news_leaks_unweighted_vs_selected"]:
    print(f"{row['p_allow_before']:.2f} -> {row['p_allow_after']:.2f}  {row['title'][:70]}")

print()
print("Still approved from an excluded mapped class:")
for row in payload["error_examples"]:
    if row["kind"] == "incorrect_approval":
        print(f"{row['p_allow']:.2f}  [{row['mapped_label']}]  {row['title'][:70]}")

## 4. Policy tradeoffs

`p_allow` is the sum of probabilities on that advertiser's allow-list. Approval is `p_allow >= threshold` (the boundary counts).

Kids & Family also excludes Gaming, Lifestyle, and Education, not only News. Thresholds other than 0.55 are illustrative. Counts use mapped labels as context proxies.


In [ ]:
policy_rows = []
for name, block in selected["policies"].items():
    t = block["at_stated_threshold"]
    policy_rows.append({
        "policy": name,
        "threshold": t["threshold"],
        "source": t["threshold_source"],
        "news_approved": f"{t['news_approvals']}/{t['news_in_split']}",
        "incorrect_over_excluded": None if t["incorrect_over_excluded"] is None else round(t["incorrect_over_excluded"], 3),
        "incorrect_over_approved": None if t["incorrect_over_approved"] is None else round(t["incorrect_over_approved"], 3),
        "retention": None if t["retention_of_allowed"] is None else round(t["retention_of_allowed"], 3),
        "approval_rate": None if t["overall_approval_rate"] is None else round(t["overall_approval_rate"], 3),
        "n_incorrect": t["incorrect_approvals"],
        "n_approved": t["n_approved"],
    })
pd.DataFrame(policy_rows)

In [ ]:
sweep = selected["policies"]["Kids & Family Brand"]["threshold_sweep"]
cuts = [float(k) for k in sweep]
incorrect = [sweep[k]["incorrect_approvals"] for k in sweep]
retention = [sweep[k]["retention_of_allowed"] for k in sweep]

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(cuts, incorrect, marker="o", color="#b91c1c", label="incorrect approvals")
ax1.set_xlabel("Kids & Family threshold")
ax1.set_ylabel("Incorrect approvals (excluded mapped labels)")
ax2 = ax1.twinx()
ax2.plot(cuts, retention, marker="s", color="#15803d", label="allowed retained")
ax2.set_ylabel("Retention of allowed mapped labels")
ax1.set_title("Development-holdout tradeoff — not a production threshold")
fig.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2)
plt.tight_layout()
plt.show()

## 5. Selected model, calibration, robustness

**Selected:** unigram TF-IDF + logistic regression with balanced class weights.

Balanced bigrams were 14 vs 15 excluded approvals — inside the 3-video tie band — and retained less eligible inventory. The simpler unigram model is the one exported to Streamlit.

**Calibration.** `p_allow` vs “mapped label is allowed” has ECE around 0.16 on this holdout. The score is not a proven probability. No calibrator was fit (any future one must use development data only).

**Robustness.** The frozen spec was retrained under a channel-disjoint split (unseen channels) and a later-publish-time split. Those checks are stress tests. They are not a reshuffle of the same videos described as a confirmation set.

**External plan.** Freeze `src/config.py` and this spec, then score a later US scrape or another country file never used here.


In [ ]:
cal = payload["calibration"]["p_allow_vs_truly_allowed"]
print("p_allow ECE", round(cal["ece"], 3), "Brier", round(cal["brier"], 3))
print()
for name, block in payload["robustness"].items():
    c = block["classification"]
    k = block["kids_family_policy"]
    warn = [cls for cls, flag in block["split"]["rare_class_warning"].items() if flag]
    print(
        name,
        "n=",
        block["split"]["n_holdout"],
        "acc",
        round(c["accuracy"], 3),
        "macro",
        round(c["macro_f1"], 3),
        "newsR",
        round(c["news_recall"], 3),
        "kids_incorrect",
        k["incorrect_approvals"],
        "news_approved",
        k["news_approvals"],
        "rare_warning",
        warn or "none",
    )

## 6. Limitations

- The development holdout was used for comparison, errors, and threshold talk.
- Mapped YouTube categories are noisy proxies (UFC highlights labelled Lifestyle).
- Gaming support is small; per-class metrics move a lot from one error.
- No unsafe-content labels exist in this scrape.
- `p_allow` is a ranking score, not a well-calibrated probability.
- There is no hosted demo. Run `py -m streamlit run app.py` locally.

Current numbers: `reports/evaluation.json`. Current decisions: the summary at the top of `DECISIONS.md`. Older D3/D4 tables in that file are a learning log, not the freeze.
